# Treasure EDU — exact 100-step GPU benchmark

This notebook uses a free Google Colab GPU when one is available. Choose **Runtime → Change runtime type → T4 GPU**, then **Runtime → Run all**. It benchmarks only 100 optimiser steps and never starts a long training run.

In [ ]:
import os, shutil, subprocess, sys
work = '/content/treasureacademyageva'
shutil.rmtree(work, ignore_errors=True)
subprocess.run(['git','clone','--depth','1','--branch','preview','https://github.com/treasureacademyageva/treasureacademyageva.git',work], check=True)
os.chdir(work + '/model')
subprocess.run([sys.executable,'-m','pip','install','-q','tokenizers>=0.20,<1','pypdf>=6,<7'], check=True)
print('Repository and lightweight dependencies are ready.')

In [ ]:
import torch
assert torch.cuda.is_available(), 'No GPU is attached. Choose Runtime > Change runtime type > T4 GPU, then Run all again.'
print('GPU:', torch.cuda.get_device_name(0))
print('VRAM GiB:', round(torch.cuda.get_device_properties(0).total_memory/2**30, 2))
subprocess.run(['nvidia-smi'], check=True)

In [ ]:
commands = [
 ['python3','scripts/build_curriculum.py'],
 ['python3','scripts/fetch_open_data.py','--strict'],
 ['python3','scripts/validate_sources.py'],
 ['python3','scripts/prepare_data.py','--config','config/micro.json'],
 ['python3','tokenizer/train_tokenizer.py','--config','config/tiny.json'],
 ['python3','scripts/prepare_data.py','--config','config/tiny.json'],
 ['python3','scripts/audit_corpus.py']
]
for command in commands:
 print('\n$', ' '.join(command), flush=True)
 subprocess.run(command, check=True)

In [ ]:
report = 'runs/tiny-gpu-benchmark.json'
command = [
 'python3','scripts/benchmark_tiny.py','--steps','100',
 '--provider','Google Colab Free','--instance','Colab assigned CUDA GPU',
 '--hourly-cost-usd','0','--target-steps','50000','--output',report
]
subprocess.run(command, check=True)
print('Exact 100-step benchmark completed. No long run was started.')

In [ ]:
import json
from google.colab import files
result = json.load(open(report))
assert result['kind'] == 'GPU' and result['benchmark']['steps'] == 100
print(json.dumps({
 'gpu': result['gpu'],
 'benchmark': result['benchmark'],
 'projection': result['projection'],
 'sample': result['sample'],
 'long_run_authorised': result['long_run_authorised']
}, indent=2))
files.download(report)